
# SVM
Cortes & Vapnik (1995): https://link.springer.com/article/10.1007/BF00994018




Twin SVM (2007): https://ieeexplore.ieee.org/document/4135685




MMA Training / Robust SVM (2020): https://arxiv.org/abs/1811.09600




Deep Kernel Learning (2016): https://arxiv.org/abs/1511.02222


# SVM Standared

In [1]:
# Arabic Sentiment Analysis — SVM (Cortes & Vapnik, 1995)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV   # for predict_proba
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# CONFIGURATION
RANDOM_STATE  = 42
TRAIN_PATH    = "/content/train_stemmed.csv"
VAL_PATH      = "/content/val_stemmed.csv"
TEST_PATH     = "/content/test_stemmed.csv"
TEXT_COL      = "clean_text_stemmed"
LABEL_COL     = "labels"
HAS_EMOJI_COL = "has_emoji"
EMOJI_CNT_COL = "emoji_count"

TFIDF_PARAMS = dict(
    analyzer      = "word",
    ngram_range   = (1, 3),
    max_features  = 50_000,
    sublinear_tf  = True,
    min_df        = 2,
    max_df        = 0.95,
    strip_accents = None,
    token_pattern = r"(?u)\b\w+\b",
)

# 1. LOAD DATA
print("=" * 60)
print("  📂 Loading Data")
print("=" * 60)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}")
print(f"Label distribution (train):\n{train_df[LABEL_COL].value_counts()}\n")

# 2. FEATURE ENGINEERING — TF-IDF + Emoji
print("=" * 60)
print("  ⚙️  Building Features")
print("=" * 60)

# Fit TF-IDF ONLY on training data — no leakage
tfidf = TfidfVectorizer(**TFIDF_PARAMS)

X_train_tfidf = tfidf.fit_transform(train_df[TEXT_COL].fillna("").astype(str))
X_val_tfidf   = tfidf.transform(val_df[TEXT_COL].fillna("").astype(str))
X_test_tfidf  = tfidf.transform(test_df[TEXT_COL].fillna("").astype(str))

print(f"TF-IDF vocab size : {len(tfidf.vocabulary_):,}")

def emoji_sparse(df):
    """Convert emoji columns to sparse matrix for hstack compatibility."""
    return csr_matrix(df[[HAS_EMOJI_COL, EMOJI_CNT_COL]].fillna(0).values.astype(float))

# LinearSVC handles sparse natively — no .toarray() needed
X_train = hstack([X_train_tfidf, emoji_sparse(train_df)])
X_val   = hstack([X_val_tfidf,   emoji_sparse(val_df)])
X_test  = hstack([X_test_tfidf,  emoji_sparse(test_df)])

print(f"Final feature shape (train) : {X_train.shape}\n")

# 3. ENCODE LABELS
le = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL])
y_val   = le.transform(val_df[LABEL_COL])
y_test  = le.transform(test_df[LABEL_COL])
print(f"Classes : {le.classes_}  →  {list(range(len(le.classes_)))}\n")

# 4. MODEL — LinearSVC (Cortes & Vapnik SVM)
# LinearSVC solves the primal SVM problem via coordinate descent (LIBLINEAR).
# It scales to large sparse feature spaces much faster than kernel SVC.
# CalibratedClassifierCV wraps it to expose predict_proba if needed.
print("=" * 60)
print("  🔷 Training: SVM — LinearSVC (Cortes & Vapnik, 1995)")
print("=" * 60)

base_svm = LinearSVC(
    C            = 1.0,        # regularisation strength (smaller = more regularised)
    loss         = "squared_hinge",
    max_iter     = 2000,
    class_weight = "balanced",
    random_state = RANDOM_STATE,
)

# Wrap with isotonic calibration to get probability outputs
clf = CalibratedClassifierCV(base_svm, cv=3, method="isotonic")
clf.fit(X_train, y_train)

# 5. EVALUATION
def evaluate(y_true, y_pred, split):
    print(f"\n{'─'*60}")
    print(f"  SVM (LinearSVC)  |  {split}")
    print(f"{'─'*60}")
    print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  F1-Score  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=le.classes_.astype(str), zero_division=0))
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n")

evaluate(y_val,  clf.predict(X_val),  "Validation Set")
evaluate(y_test, clf.predict(X_test), "Test Set")

  📂 Loading Data
Train : 5,067 | Val : 1,086 | Test : 1,086
Label distribution (train):
labels
1    2536
0    2531
Name: count, dtype: int64

  ⚙️  Building Features
TF-IDF vocab size : 7,726
Final feature shape (train) : (5067, 7728)

Classes : [0 1]  →  [0, 1]

  🔷 Training: SVM — LinearSVC (Cortes & Vapnik, 1995)

────────────────────────────────────────────────────────────
  SVM (LinearSVC)  |  Validation Set
────────────────────────────────────────────────────────────
  Accuracy  : 0.8306
  Precision : 0.8310
  Recall    : 0.8306
  F1-Score  : 0.8305

  Classification Report:

              precision    recall  f1-score   support

           0       0.84      0.81      0.83       543
           1       0.82      0.85      0.83       543

    accuracy                           0.83      1086
   macro avg       0.83      0.83      0.83      1086
weighted avg       0.83      0.83      0.83      1086

  Confusion Matrix:
[[441 102]
 [ 82 461]]


───────────────────────────────────────

# Twin SVM

In [2]:
# Arabic Sentiment Analysis — Twin SVM (TSVM)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# ── Try ThunderSVM first (GPU/CPU TSVM), fall back to libtsvm ────────────────
try:
    from thundersvm import SVC as ThunderSVC
    BACKEND = "thundersvm"
except ImportError:
    try:
        from libtsvm.estimators import TSVM
        BACKEND = "libtsvm"
    except ImportError:
        BACKEND = "sklearn_fallback"
        print("⚠️  Neither thundersvm nor libtsvm found.")
        print("   Falling back to sklearn SVC with RBF kernel (not true TSVM).")
        print("   Install TSVM with: pip install thunder-svm  OR  pip install libtsvm\n")
        from sklearn.svm import SVC

# CONFIGURATION
RANDOM_STATE  = 42
TRAIN_PATH    = "/content/train_stemmed.csv"
VAL_PATH      = "/content/val_stemmed.csv"
TEST_PATH     = "/content/test_stemmed.csv"
TEXT_COL      = "clean_text_stemmed"
LABEL_COL     = "labels"
HAS_EMOJI_COL = "has_emoji"
EMOJI_CNT_COL = "emoji_count"

TFIDF_PARAMS = dict(
    analyzer      = "word",
    ngram_range   = (1, 3),
    max_features  = 30_000,   # reduced: TSVM is memory-heavier than LinearSVC
    sublinear_tf  = True,
    min_df        = 2,
    max_df        = 0.95,
    strip_accents = None,
    token_pattern = r"(?u)\b\w+\b",
)

# 1. LOAD DATA
print("=" * 60)
print("  📂 Loading Data")
print("=" * 60)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}")
print(f"Label distribution (train):\n{train_df[LABEL_COL].value_counts()}\n")

# 2. FEATURE ENGINEERING — TF-IDF + Emoji
print("=" * 60)
print("  ⚙️  Building Features")
print("=" * 60)

tfidf = TfidfVectorizer(**TFIDF_PARAMS)

X_train_tfidf = tfidf.fit_transform(train_df[TEXT_COL].fillna("").astype(str))
X_val_tfidf   = tfidf.transform(val_df[TEXT_COL].fillna("").astype(str))
X_test_tfidf  = tfidf.transform(test_df[TEXT_COL].fillna("").astype(str))

print(f"TF-IDF vocab size : {len(tfidf.vocabulary_):,}")

def emoji_sparse(df):
    return csr_matrix(df[[HAS_EMOJI_COL, EMOJI_CNT_COL]].fillna(0).values.astype(float))

# TSVM requires dense arrays
X_train = hstack([X_train_tfidf, emoji_sparse(train_df)]).toarray()
X_val   = hstack([X_val_tfidf,   emoji_sparse(val_df)]).toarray()
X_test  = hstack([X_test_tfidf,  emoji_sparse(test_df)]).toarray()

print(f"Final feature shape (train) : {X_train.shape}\n")

# 3. ENCODE LABELS
le = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL])
y_val   = le.transform(val_df[LABEL_COL])
y_test  = le.transform(test_df[LABEL_COL])
print(f"Classes : {le.classes_}  →  {list(range(len(le.classes_)))}\n")

# 4. MODEL — Twin SVM
print("=" * 60)
print(f"  🔷 Training: Twin SVM  [{BACKEND}]")
print("=" * 60)

if BACKEND == "thundersvm":
    # ThunderSVM is a GPU-accelerated SVM library with TSVM support
    clf = ThunderSVC(
        kernel       = "linear",
        C            = 1.0,
        class_weight = "balanced",
    )
    clf.fit(X_train, y_train)

elif BACKEND == "libtsvm":
    # libtsvm is the reference Python Twin SVM implementation
    # Labels must be -1 / +1 for binary TSVM
    y_tsvm = np.where(y_train == 0, -1, 1)
    clf = TSVM(kernel="linear", C1=1.0, C2=1.0)
    clf.fit(X_train, y_tsvm)

    # Re-map predictions back to 0/1 for unified evaluation
    def predict_tsvm(X):
        raw = clf.predict(X)
        return np.where(raw == -1, 0, 1)

else:  # sklearn fallback
    from sklearn.svm import SVC
    clf = SVC(
        kernel       = "linear",
        C            = 1.0,
        class_weight = "balanced",
        random_state = RANDOM_STATE,
    )
    clf.fit(X_train, y_train)

# 5. EVALUATION
def predict(X):
    if BACKEND == "libtsvm":
        return predict_tsvm(X)
    return clf.predict(X)

def evaluate(y_true, y_pred, split):
    print(f"\n{'─'*60}")
    print(f"  Twin SVM  |  {split}")
    print(f"{'─'*60}")
    print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  F1-Score  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=le.classes_.astype(str), zero_division=0))
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n")

evaluate(y_val,  predict(X_val),  "Validation Set")
evaluate(y_test, predict(X_test), "Test Set")

⚠️  Neither thundersvm nor libtsvm found.
   Falling back to sklearn SVC with RBF kernel (not true TSVM).
   Install TSVM with: pip install thunder-svm  OR  pip install libtsvm

  📂 Loading Data
Train : 5,067 | Val : 1,086 | Test : 1,086
Label distribution (train):
labels
1    2536
0    2531
Name: count, dtype: int64

  ⚙️  Building Features
TF-IDF vocab size : 7,726
Final feature shape (train) : (5067, 7728)

Classes : [0 1]  →  [0, 1]

  🔷 Training: Twin SVM  [sklearn_fallback]

────────────────────────────────────────────────────────────
  Twin SVM  |  Validation Set
────────────────────────────────────────────────────────────
  Accuracy  : 0.8389
  Precision : 0.8391
  Recall    : 0.8389
  F1-Score  : 0.8388

  Classification Report:

              precision    recall  f1-score   support

           0       0.85      0.83      0.84       543
           1       0.83      0.85      0.84       543

    accuracy                           0.84      1086
   macro avg       0.84      0.84

# Robust SVM

In [3]:
# Arabic Sentiment Analysis — Robust SVM / MMA Training (2020)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix, vstack

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# CONFIGURATION
RANDOM_STATE  = 42
TRAIN_PATH    = "/content/train_stemmed.csv"
VAL_PATH      = "/content/val_stemmed.csv"
TEST_PATH     = "/content/test_stemmed.csv"
TEXT_COL      = "clean_text_stemmed"
LABEL_COL     = "labels"
HAS_EMOJI_COL = "has_emoji"
EMOJI_CNT_COL = "emoji_count"

# MMA-specific: perturbation budget (epsilon) in L-inf norm
EPSILON = 0.01

TFIDF_PARAMS = dict(
    analyzer      = "word",
    ngram_range   = (1, 3),
    max_features  = 50_000,
    sublinear_tf  = True,
    min_df        = 2,
    max_df        = 0.95,
    strip_accents = None,
    token_pattern = r"(?u)\b\w+\b",
)

# 1. LOAD DATA
print("=" * 60)
print("  📂 Loading Data")
print("=" * 60)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}")
print(f"Label distribution (train):\n{train_df[LABEL_COL].value_counts()}\n")

# 2. FEATURE ENGINEERING — TF-IDF + Emoji
print("=" * 60)
print("  ⚙️  Building Features")
print("=" * 60)

tfidf = TfidfVectorizer(**TFIDF_PARAMS)

X_train_tfidf = tfidf.fit_transform(train_df[TEXT_COL].fillna("").astype(str))
X_val_tfidf   = tfidf.transform(val_df[TEXT_COL].fillna("").astype(str))
X_test_tfidf  = tfidf.transform(test_df[TEXT_COL].fillna("").astype(str))

print(f"TF-IDF vocab size : {len(tfidf.vocabulary_):,}")

def emoji_sparse(df):
    return csr_matrix(df[[HAS_EMOJI_COL, EMOJI_CNT_COL]].fillna(0).values.astype(float))

X_train = hstack([X_train_tfidf, emoji_sparse(train_df)])
X_val   = hstack([X_val_tfidf,   emoji_sparse(val_df)])
X_test  = hstack([X_test_tfidf,  emoji_sparse(test_df)])

print(f"Final feature shape (train) : {X_train.shape}\n")

# 3. ENCODE LABELS
le = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL])
y_val   = le.transform(val_df[LABEL_COL])
y_test  = le.transform(test_df[LABEL_COL])
print(f"Classes : {le.classes_}  →  {list(range(len(le.classes_)))}\n")

# 4a. PHASE 1 — Fit Base SVM
print("=" * 60)
print("  🔷 Phase 1: Fitting base LinearSVC …")
print("=" * 60)

base_svm = LinearSVC(
    C            = 1.0,
    loss         = "squared_hinge",
    max_iter     = 2000,
    class_weight = "balanced",
    random_state = RANDOM_STATE,
)
base_svm.fit(X_train, y_train)

# 4b. PHASE 2 — Generate Adversarial Perturbations (FGSM approximation)
# For the linear SVM:  sign of the weight vector w gives the gradient
# direction of the hinge loss w.r.t. the input.
# FGSM perturbation:  x_adv = x + epsilon * sign(w * y_true)
# where y_true is in {-1, +1}.
print("  ⚔️  Phase 2: Generating adversarial samples (MMA/FGSM) …")

# Map labels to {-1, +1} for the gradient sign computation
y_signed = np.where(y_train == 0, -1.0, 1.0)

# Weight vector shape: (n_classes-1, n_features) for binary
w = base_svm.coef_[0]   # shape: (n_features,)

# Sparse-compatible perturbation: add epsilon * sign(w) * label_sign
# We work on the dense perturbation only (small matrix, n_samples x n_feat)
sign_w = np.sign(w)  # (n_features,)

# Perturbation for each sample: EPSILON * sign(w) * (-y_signed) pushes
# samples toward the wrong side (adversarial direction)
# Dense perturbation matrix: (n_samples, n_features)
perturbation = EPSILON * np.outer(-y_signed, sign_w)  # (n, d)

# Add perturbation to original sparse features → convert X_train to dense
X_train_dense = X_train.toarray()
X_adv         = X_train_dense + perturbation

# Concatenate original + adversarial data
X_augmented = np.vstack([X_train_dense, X_adv])
y_augmented = np.concatenate([y_train, y_train])

print(f"   Augmented training set size : {X_augmented.shape[0]:,}\n")

# 4c. PHASE 3 — Refit on Augmented Data (Robust SVM)
print("  🔷 Phase 3: Refitting Robust SVM on augmented data …")

robust_svm = LinearSVC(
    C            = 1.0,
    loss         = "squared_hinge",
    max_iter     = 2000,
    class_weight = "balanced",
    random_state = RANDOM_STATE,
)

clf = CalibratedClassifierCV(robust_svm, cv=3, method="isotonic")
clf.fit(X_augmented, y_augmented)

# 5. EVALUATION
def evaluate(y_true, y_pred, split):
    print(f"\n{'─'*60}")
    print(f"  Robust SVM (MMA)  |  {split}")
    print(f"{'─'*60}")
    print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  F1-Score  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=le.classes_.astype(str), zero_division=0))
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n")

evaluate(y_val,  clf.predict(X_val.toarray()),  "Validation Set")
evaluate(y_test, clf.predict(X_test.toarray()), "Test Set")

  📂 Loading Data
Train : 5,067 | Val : 1,086 | Test : 1,086
Label distribution (train):
labels
1    2536
0    2531
Name: count, dtype: int64

  ⚙️  Building Features
TF-IDF vocab size : 7,726
Final feature shape (train) : (5067, 7728)

Classes : [0 1]  →  [0, 1]

  🔷 Phase 1: Fitting base LinearSVC …
  ⚔️  Phase 2: Generating adversarial samples (MMA/FGSM) …
   Augmented training set size : 10,134

  🔷 Phase 3: Refitting Robust SVM on augmented data …

────────────────────────────────────────────────────────────
  Robust SVM (MMA)  |  Validation Set
────────────────────────────────────────────────────────────
  Accuracy  : 0.8361
  Precision : 0.8364
  Recall    : 0.8361
  F1-Score  : 0.8361

  Classification Report:

              precision    recall  f1-score   support

           0       0.85      0.82      0.83       543
           1       0.83      0.85      0.84       543

    accuracy                           0.84      1086
   macro avg       0.84      0.84      0.84      1086


# Deep Kernel Learning

In [4]:
# Arabic Sentiment Analysis — Deep Kernel Learning (DKL)


import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# CONFIGURATION
RANDOM_STATE  = 42
TRAIN_PATH    = "/content/train_stemmed.csv"
VAL_PATH      = "/content/val_stemmed.csv"
TEST_PATH     = "/content/test_stemmed.csv"
TEXT_COL      = "clean_text_stemmed"
LABEL_COL     = "labels"
HAS_EMOJI_COL = "has_emoji"
EMOJI_CNT_COL = "emoji_count"

# MLP hidden layer sizes: each tuple = (layer1_units, layer2_units, ...)
# The output of the last hidden layer is used as the deep kernel embedding
MLP_HIDDEN    = (512, 128)
MLP_MAX_ITER  = 200

# SVC kernel applied on top of the deep embedding
SVC_C         = 5.0
SVC_GAMMA     = "scale"

TFIDF_PARAMS = dict(
    analyzer      = "word",
    ngram_range   = (1, 3),
    max_features  = 50_000,
    sublinear_tf  = True,
    min_df        = 2,
    max_df        = 0.95,
    strip_accents = None,
    token_pattern = r"(?u)\b\w+\b",
)

# 1. LOAD DATA
print("=" * 60)
print("  📂 Loading Data")
print("=" * 60)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}")
print(f"Label distribution (train):\n{train_df[LABEL_COL].value_counts()}\n")

# 2. FEATURE ENGINEERING — TF-IDF + Emoji
print("=" * 60)
print("  ⚙️  Building TF-IDF Features")
print("=" * 60)

tfidf = TfidfVectorizer(**TFIDF_PARAMS)

X_train_tfidf = tfidf.fit_transform(train_df[TEXT_COL].fillna("").astype(str))
X_val_tfidf   = tfidf.transform(val_df[TEXT_COL].fillna("").astype(str))
X_test_tfidf  = tfidf.transform(test_df[TEXT_COL].fillna("").astype(str))

print(f"TF-IDF vocab size : {len(tfidf.vocabulary_):,}")

def emoji_sparse(df):
    return csr_matrix(df[[HAS_EMOJI_COL, EMOJI_CNT_COL]].fillna(0).values.astype(float))

# Convert to dense — MLP requires dense arrays
X_train_raw = hstack([X_train_tfidf, emoji_sparse(train_df)]).toarray()
X_val_raw   = hstack([X_val_tfidf,   emoji_sparse(val_df)]).toarray()
X_test_raw  = hstack([X_test_tfidf,  emoji_sparse(test_df)]).toarray()

print(f"Final feature shape (train) : {X_train_raw.shape}\n")

# 3. ENCODE LABELS
le = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL])
y_val   = le.transform(val_df[LABEL_COL])
y_test  = le.transform(test_df[LABEL_COL])
print(f"Classes : {le.classes_}  →  {list(range(len(le.classes_)))}\n")

# 4a. DEEP KERNEL LEARNING — Step 1: Train MLP Feature Extractor
# The MLP learns a non-linear mapping from the raw TF-IDF space to a
# low-dimensional embedding that is discriminative for sentiment.
print("=" * 60)
print("  🧠 Step 1: Training MLP Feature Extractor …")
print("=" * 60)

mlp = MLPClassifier(
    hidden_layer_sizes = MLP_HIDDEN,
    activation         = "relu",
    solver             = "adam",
    alpha              = 1e-4,           # L2 regularisation
    batch_size         = 256,
    learning_rate_init = 1e-3,
    max_iter           = MLP_MAX_ITER,
    early_stopping     = True,
    validation_fraction= 0.1,
    n_iter_no_change   = 10,
    random_state       = RANDOM_STATE,
    verbose            = False,
)
mlp.fit(X_train_raw, y_train)
print(f"   MLP training complete. Best val score : {mlp.best_validation_score_:.4f}\n")

# 4b. DEEP KERNEL LEARNING — Step 2: Extract Penultimate Layer Embeddings
# We use the activation of the last hidden layer as our deep kernel embedding.
# This is equivalent to Φ(x) in the DKL framework.
def extract_embedding(mlp_model, X: np.ndarray) -> np.ndarray:
    """Forward pass through all hidden layers; return last hidden activation."""
    activations = [X]
    for i, (W, b) in enumerate(zip(mlp_model.coefs_[:-1], mlp_model.intercepts_[:-1])):
        Z = activations[-1] @ W + b
        Z = np.maximum(0, Z)  # ReLU
        activations.append(Z)
    return activations[-1]  # penultimate layer output

print("  🔗 Step 2: Extracting deep embeddings …")
X_train_emb = extract_embedding(mlp, X_train_raw)
X_val_emb   = extract_embedding(mlp, X_val_raw)
X_test_emb  = extract_embedding(mlp, X_test_raw)
print(f"   Embedding dimension : {X_train_emb.shape[1]}")

# 4c. DEEP KERNEL LEARNING — Step 3: Train RBF-SVC on Deep Embeddings
# The RBF kernel applied to deep embeddings approximates the Deep Kernel
# K(x, x') = k(Φ(x), Φ(x')), where Φ is the neural network.
print("\n  🔷 Step 3: Training RBF-SVC on deep embeddings …")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_emb)
X_val_scaled   = scaler.transform(X_val_emb)
X_test_scaled  = scaler.transform(X_test_emb)

clf = SVC(
    kernel       = "rbf",
    C            = SVC_C,
    gamma        = SVC_GAMMA,
    class_weight = "balanced",
    random_state = RANDOM_STATE,
    probability  = True,
)
clf.fit(X_train_scaled, y_train)

# 5. EVALUATION
def evaluate(y_true, y_pred, split):
    print(f"\n{'─'*60}")
    print(f"  Deep Kernel Learning SVM  |  {split}")
    print(f"{'─'*60}")
    print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  F1-Score  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=le.classes_.astype(str), zero_division=0))
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n")

evaluate(y_val,  clf.predict(X_val_scaled),  "Validation Set")
evaluate(y_test, clf.predict(X_test_scaled), "Test Set")

  📂 Loading Data
Train : 5,067 | Val : 1,086 | Test : 1,086
Label distribution (train):
labels
1    2536
0    2531
Name: count, dtype: int64

  ⚙️  Building TF-IDF Features
TF-IDF vocab size : 7,726
Final feature shape (train) : (5067, 7728)

Classes : [0 1]  →  [0, 1]

  🧠 Step 1: Training MLP Feature Extractor …
   MLP training complete. Best val score : 0.8442

  🔗 Step 2: Extracting deep embeddings …
   Embedding dimension : 128

  🔷 Step 3: Training RBF-SVC on deep embeddings …

────────────────────────────────────────────────────────────
  Deep Kernel Learning SVM  |  Validation Set
────────────────────────────────────────────────────────────
  Accuracy  : 0.8379
  Precision : 0.8387
  Recall    : 0.8379
  F1-Score  : 0.8378

  Classification Report:

              precision    recall  f1-score   support

           0       0.85      0.81      0.83       543
           1       0.82      0.86      0.84       543

    accuracy                           0.84      1086
   macro avg  